# Análise Exploratória (EDA)

**Projeto:** Social Wave — Otimização de Campanhas de Marketing  
**Fase:** Diagnóstico (Fase 1)  
**Objetivo:** Entender o passado antes de propor o futuro  

---

## Perguntas que este notebook responde

| # | Pergunta | O que descobrimos | Seção |
|---|----------|-------------------|-------|
| **1.1** | Qual o **CPA médio por canal** e como ele evoluiu ao longo do tempo? | Canais eficientes vs. ineficientes | 3.1 |
| **1.2** | Qual o **share de gasto** e **share de conversões** por canal hoje? | Desbalanceamento entre investimento e retorno | 3.2 |
| **1.3** | Existe **correlação entre aumento de gasto e elevação de CPA** dentro de cada canal? | Sinais de retornos decrescentes | 3.3 |
| **1.4** | Qual o **CTR e taxa de conversão** por canal? | Onde o funil 'vaza' | 3.4 |

---

## Dados de Entrada
Base limpa do notebook `notebook_00_contexto_dicionario_e_tratamento.ipynb`  
Período: Outubro a Dezembro de 2023  
Registros: [será preenchido após carga]  
Canais: 6 (Meta, Google, YouTube, Twitter, TikTok, LinkedIn)

---

## Entregáveis
- Tabela resumo por canal (CPA, CTR, Taxa de Conversão, Share)
- Gráficos: evolução temporal, distribuição, correlações
- Insights iniciais para o modelo de otimização

## Setup e Carregamento da Base processada no Notebook 00

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# --- Configuracoes visuais ---
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# --- Configuracoes de exibicao ---
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# --- Seed para reprodutibilidade ---
np.random.seed(42)

# ============================================================
# CARREGAR BASE PROCESSADA (PICKLE)
# ============================================================

PICKLE_PATH = r'C:\Users\ricar\Documents\projetos\social-wave-campaign-optimization\data\campanhas_base_processada.pkl'

try:
    df = pd.read_pickle(PICKLE_PATH)
    
    print('=' * 60)
    print('📊 BASE PROCESSADA CARREGADA')
    print('=' * 60)
    print(f'Fonte: {PICKLE_PATH}')
    print(f'Dimensoes: {df.shape[0]:,} registros x {df.shape[1]} colunas')
    print(f'Periodo: {df['Data da Coleta'].min()} a {df['Data da Coleta'].max()}')
    
    print(f'\n✅ Tipos preservados:')
    print(f'   • Data da Coleta: {df['Data da Coleta'].dtype}')
    print(f'   • Canal: {df['Canal'].dtype}')
    print(f'   • Gasto: {df['Gasto'].dtype}')
    print(f'   • Conversao: {df['Conversao'].dtype}')
    
    print(f'\n📋 Colunas disponiveis:')
    print(f'   {', '.join(df.columns.tolist())}')
    
except FileNotFoundError:
    print(f'❌ Base nao encontrada: {PICKLE_PATH}')
    print('   Execute o Notebook 00 primeiro para gerar a base limpa.')
    raise

print(f'\n📊 Primeiras linhas:')
display(df.head())

📊 BASE PROCESSADA CARREGADA
Fonte: C:\Users\ricar\Documents\projetos\social-wave-campaign-optimization\data\campanhas_base_processada.pkl
Dimensoes: 23,315 registros x 9 colunas
Periodo: 2023-10-01 11:00:00 a 2023-12-24 23:00:00

✅ Tipos preservados:
   • Data da Coleta: datetime64[us]
   • Canal: str
   • Gasto: float64
   • Conversao: int64

📋 Colunas disponiveis:
   ID, Canal, Data da Coleta, Impressoes Disponiveis, Consultas Correspondentes, Impressoes, Cliques, Gasto, Conversao

📊 Primeiras linhas:


,ID,Canal,Data da Coleta,Impressoes Disponiveis,Consultas Correspondentes,Impressoes,Cliques,Gasto,Conversao
0,ADSXJ0000001,Canal1,2023-11-02 03:00:00,82132,61776,43673,5,37.76,2
1,ADSXJ0000002,Canal1,2023-11-02 03:00:00,41889,33512,23919,4,18.76,2
2,ADSXJ0000003,Canal2,2023-12-18 04:00:00,2438114,1611096,1215019,1314,2481.57,44
3,ADSXJ0000004,Canal2,2023-12-19 04:00:00,2429576,1608711,1212549,1343,2513.02,36
4,ADSXJ0000005,Canal2,2023-12-18 03:00:00,2863336,1929554,1456197,1566,2912.62,30


## Cálculo de Métricas Derivadas

Objetivo: Transformar dados brutos em KPIs de marketing.

Fórmulas: CTR, Taxa de Conversão, CPC, CPA, CPM

In [5]:
print('=' * 60)
print('📐 CÁLCULO DE MÉTRICAS DERIVADAS')
print('=' * 60)

# ============================================================
# 1. MÉTRICAS DO FUNIL (Taxas)
# ============================================================

print('\n' + '-' * 60)
print('1. MÉTRICAS DO FUNIL')
print('-' * 60)

# CTR (Click-Through Rate): % de impressões que viraram cliques
# Indica: Atratividade do criativo/impacto do anúncio
df['CTR'] = (df['Cliques'] / df['Impressoes'] * 100).round(2)

# Taxa de Conversão: % de cliques que viraram conversões
# Indica: Eficiência do funil pós-clique (landing page, oferta)
df['Taxa_Conversao'] = (df['Conversao'] / df['Cliques'] * 100).round(2)

print('✅ CTR calculado: (Cliques / Impressoes) x 100')
print('✅ Taxa_Conversao calculada: (Conversao / Cliques) x 100')

# ============================================================
# 2. MÉTRICAS DE CUSTO
# ============================================================

print('\n' + '-' * 60)
print('2. MÉTRICAS DE CUSTO')
print('-' * 60)

# CPC (Custo por Clique): Quanto paga por cada clique
# Indica: Custo do tráfego qualificado
df['CPC'] = (df['Gasto'] / df['Cliques']).round(2)

# CPA (Custo Por Aquisição): Quanto paga por cada conversão
# Indica: Eficiência de aquisição — KPI PRINCIPAL DO PROJETO
df['CPA'] = (df['Gasto'] / df['Conversao']).round(2)

# CPM (Custo por Mil Impressões): Custo de alcance
# Indica: Eficiência de veiculação
df['CPM'] = ((df['Gasto'] / df['Impressoes']) * 1000).round(2)

print('✅ CPC calculado: Gasto / Cliques')
print('✅ CPA calculado: Gasto / Conversao  [KPI PRINCIPAL]')
print('✅ CPM calculado: (Gasto / Impressoes) x 1000')

# ============================================================
# 3. TRATAMENTO DE DIVISÃO POR ZERO
# ============================================================

print('\n' + '-' * 60)
print('3. TRATAMENTO DE DIVISÃO POR ZERO')
print('-' * 60)

# Onde Conversao = 0, CPA = infinito — vamos identificar
cpa_infinito = np.isinf(df['CPA']).sum()
print(f'⚠️ Registros com CPA infinito (Conversao=0): {cpa_infinito}')

# Flag para identificar registros sem conversão
df['Sem_Conversao'] = df['Conversao'] == 0

print(f'✅ Flag Sem_Conversao criada para análise separada')
print(f'   Estes registros serão excluídos do cálculo de CPA médio')
print(f'   Mas mantidos na análise de CTR e CPC')

# ============================================================
# 4. MÉTRICAS DE ALCANCE
# ============================================================

print('\n' + '-' * 60)
print('4. MÉTRICAS DE ALCANCE')
print('-' * 60)

# Taxa de Impressão: % do inventário utilizado
df['Taxa_Impressao'] = (df['Impressoes'] / df['Impressoes Disponiveis'] * 100).round(2)

print('✅ Taxa_Impressao calculada: (Impressoes / Impressoes Disponiveis) x 100')

# ============================================================
# 5. COMPONENTES TEMPORAIS (se ainda não existirem)
# ============================================================

print('\n' + '-' * 60)
print('5. COMPONENTES TEMPORAIS')
print('-' * 60)

if 'Data' not in df.columns:
    df['Data'] = df['Data da Coleta'].dt.date
    print('✅ Data extraída da coluna Data da Coleta')

if 'Ano_Mes' not in df.columns:
    df['Ano_Mes'] = df['Data da Coleta'].dt.to_period('M')
    print('✅ Ano_Mes extraído da Coluna Data da Coleta')

if 'Dia_Semana' not in df.columns:
    df['Dia_Semana'] = df['Data da Coleta'].dt.day_name()
    print('✅ Dia_Semana extraído da coluna Data da Coleta')

# ============================================================
# 6. RESUMO DAS NOVAS COLUNAS
# ============================================================

print('\n' + '=' * 60)
print('📋 RESUMO DAS MÉTRICAS CRIADAS')
print('=' * 60)

metricas_novas = ['CTR', 'Taxa_Conversao', 'CPC', 'CPA', 'CPM', 'Taxa_Impressao']
print(f'\nNovas colunas: {len(metricas_novas)}')
for m in metricas_novas:
    print(f'   • {m}')

print(f'\n📊 Preview das métricas (primeiros 5 registros):')
display(df[['Canal', 'Gasto', 'Conversao', 'CTR', 'Taxa_Conversao', 'CPA', 'CPC']].head())

print(f'\n📊 Estatísticas descritivas das métricas:')
display(df[metricas_novas].describe().round(2))

📐 CÁLCULO DE MÉTRICAS DERIVADAS

------------------------------------------------------------
1. MÉTRICAS DO FUNIL
------------------------------------------------------------
✅ CTR calculado: (Cliques / Impressoes) x 100
✅ Taxa_Conversao calculada: (Conversao / Cliques) x 100

------------------------------------------------------------
2. MÉTRICAS DE CUSTO
------------------------------------------------------------
✅ CPC calculado: Gasto / Cliques
✅ CPA calculado: Gasto / Conversao  [KPI PRINCIPAL]
✅ CPM calculado: (Gasto / Impressoes) x 1000

------------------------------------------------------------
3. TRATAMENTO DE DIVISÃO POR ZERO
------------------------------------------------------------
⚠️ Registros com CPA infinito (Conversao=0): 700
✅ Flag Sem_Conversao criada para análise separada
   Estes registros serão excluídos do cálculo de CPA médio
   Mas mantidos na análise de CTR e CPC

------------------------------------------------------------
4. MÉTRICAS DE ALCANCE
--------

,Canal,Gasto,Conversao,CTR,Taxa_Conversao,CPA,CPC
0,Canal1,37.76,2,0.01,40.00,18.88,7.55
1,Canal1,18.76,2,0.02,50.00,9.38,4.69
2,Canal2,2481.57,44,0.11,3.35,56.40,1.89
3,Canal2,2513.02,36,0.11,2.68,69.81,1.87
4,Canal2,2912.62,30,0.11,1.92,97.09,1.86



📊 Estatísticas descritivas das métricas:


,CTR,Taxa_Conversao,CPC,CPA,CPM,Taxa_Impressao
count,23315.00,23315.00,23315.00,23315.00,23315.00,23315.00
mean,9.50,3.55,0.33,inf,9.09,44.11
std,10.20,9.33,0.33,NaN,9.63,14.46
min,0.01,0.00,0.00,0.00,0.00,3.70
25%,0.30,1.41,0.09,3.64,1.91,34.30
50%,10.62,2.47,0.13,8.63,9.01,47.65
75%,15.45,3.06,0.53,26.94,14.20,54.37
max,100.00,100.00,7.55,inf,743.10,100.00


OBS.: 

⚠️ Sobre o CPA infinito
Quando Conversao = 0, a divisão Gasto / 0 resulta em inf. Isso é matematicamente correto (gastou e não converteu = custo infinito por conversão), mas para análise agregada por canal usaremos média ponderada ou filtro de registros com conversão.

## Tabela Executiva por Canal

Agrupamento os dados por canal e calculo das métricas agregadas. 

Esta tabela é o diagnóstico principal — ela mostra, em uma única visão, qual canal é eficiente, qual é caro, onde está o desbalanceamento.

In [12]:
print('=' * 60)
print('📊 TABELA EXECUTIVA POR CANAL')
print('=' * 60)

# ============================================================
# MAPEAMENTO DE CANAIS
# ============================================================

print('\n' + '-' * 60)
print('MAPEAMENTO DE CANAIS')
print('-' * 60)

mapeamento_canais = {
    'Canal1': 'Meta Ads',
    'Canal2': 'Google Search',
    'Canal3': 'YouTube Ads',
    'Canal4': 'Twitter Ads',
    'Canal5': 'TikTok',
    'Canal6': 'LinkedIn Ads'
}

df['Canal_Nome'] = df['Canal'].map(mapeamento_canais)

print('✅ Canal_Nome criado:')
for codigo, nome in mapeamento_canais.items():
    print(f'   {codigo} -> {nome}')

# ============================================================
# AGREGAÇÃO POR CANAL
# ============================================================

# Agrupar por canal e somar métricas brutas
resumo = df.groupby('Canal_Nome').agg({
    'Gasto': 'sum',
    'Conversao': 'sum',
    'Impressoes': 'sum',
    'Cliques': 'sum',
    'ID': 'count'  # quantidade de registros/campanhas
}).round(2)

# Renomear colunas
resumo.columns = ['Gasto_Total', 'Conversao_Total', 'Impressoes_Total', 'Cliques_Total', 'Campanhas']

# ============================================================
# CÁLCULO DE MÉTRICAS AGREGADAS
# ============================================================

# CTR agregado: cliques totais / impressões totais
resumo['CTR'] = (resumo['Cliques_Total'] / resumo['Impressoes_Total'] * 100).round(2)

# Taxa de Conversão agregada: conversões totais / cliques totais
resumo['Taxa_Conversao'] = (resumo['Conversao_Total'] / resumo['Cliques_Total'] * 100).round(2)

# CPA agregado: gasto total / conversões totais (média ponderada correta)
resumo['CPA'] = (resumo['Gasto_Total'] / resumo['Conversao_Total']).round(2)

# CPC agregado: gasto total / cliques totais
resumo['CPC'] = (resumo['Gasto_Total'] / resumo['Cliques_Total']).round(2)

# CPM agregado
resumo['CPM'] = ((resumo['Gasto_Total'] / resumo['Impressoes_Total']) * 1000).round(2)

# ============================================================
# SHARES (PESOS RELATIVOS)
# ============================================================

print('\n' + '-' * 60)
print('SHARES DE GASTO E CONVERSÃO')
print('-' * 60)

gasto_total_geral = resumo['Gasto_Total'].sum()
conversao_total_geral = resumo['Conversao_Total'].sum()

resumo['Share_Gasto'] = (resumo['Gasto_Total'] / gasto_total_geral * 100).round(2)
resumo['Share_Conversao'] = (resumo['Conversao_Total'] / conversao_total_geral * 100).round(2)

# Diferença entre share de gasto e share de conversão
# Positivo = sobre-investido | Negativo = sub-investido
resumo['Diferenca_Share'] = (resumo['Share_Gasto'] - resumo['Share_Conversao']).round(2)

print('✅ Share_Gasto: % do investimento total por canal')
print('✅ Share_Conversao: % das conversões totais por canal')
print('✅ Diferenca_Share: desbalanceamento (positivo = sobre-investido)')

# ============================================================
# ORDENAR E EXIBIR
# ============================================================

print('\n' + '-' * 60)
print('TABELA EXECUTIVA')
print('-' * 60)

# Ordenar por CPA (menor = mais eficiente)
resumo = resumo.sort_values('CPA')

# Selecionar colunas para exibição
colunas_display = [
    'Gasto_Total', 'Conversao_Total', 'CPA', 'CTR', 
    'Taxa_Conversao', 'Share_Gasto', 'Share_Conversao', 'Diferenca_Share'
]

print('\n📋 Ordenado por CPA (menor = mais eficiente):')
print(resumo[colunas_display].to_string())

# ============================================================
# INSIGHTS AUTOMÁTICOS
# ============================================================

print('\n' + '=' * 60)
print('INSIGHTS INICIAIS')
print('=' * 60)

# Canal mais eficiente
canal_top = resumo['CPA'].idxmin()
cpa_top = resumo.loc[canal_top, 'CPA']

# Canal menos eficiente
canal_bottom = resumo['CPA'].idxmax()
cpa_bottom = resumo.loc[canal_bottom, 'CPA']

# Canal mais investido
canal_maior_gasto = resumo['Share_Gasto'].idxmax()

# Canal com maior desbalanceamento positivo (sobre-investido)
canal_sobre_investido = resumo['Diferenca_Share'].idxmax()
diff_sobre = resumo.loc[canal_sobre_investido, 'Diferenca_Share']

# Canal com maior desbalanceamento negativo (sub-investido)
canal_sub_investido = resumo['Diferenca_Share'].idxmin()
diff_sub = resumo.loc[canal_sub_investido, 'Diferenca_Share']

print(f'\nMAIS EFICIENTE: {canal_top} (CPA: ${cpa_top:.2f})')
print(f'MENOS EFICIENTE: {canal_bottom} (CPA: ${cpa_bottom:.2f})')
print(f'   Diferença de eficiência: {((cpa_bottom / cpa_top) - 1) * 100:.1f}% mais caro')

print(f'\nMAIOR INVESTIMENTO: {canal_maior_gasto} ({resumo.loc[canal_maior_gasto, 'Share_Gasto']:.1f}% do gasto total)')

print(f'\nDESBALANCEAMENTO:')
print(f'   SOBRE-INVESTIDO: {canal_sobre_investido} (+{diff_sobre:.1f}%)')
print(f'      → Gasta {resumo.loc[canal_sobre_investido, 'Share_Gasto']:.1f}% do orçamento para {resumo.loc[canal_sobre_investido, 'Share_Conversao']:.1f}% das conversões')
print(f'   SUB-INVESTIDO: {canal_sub_investido} ({diff_sub:.1f}%)')
print(f'      → Gasta {resumo.loc[canal_sub_investido, 'Share_Gasto']:.1f}% do orçamento para {resumo.loc[canal_sub_investido, 'Share_Conversao']:.1f}% das conversões')

# ============================================================
# SALVAR RESUMO PARA USO FUTURO
# ============================================================

print('\n' + '-' * 60)
print('SALVAR RESUMO')
print('-' * 60)

# Usar raw string (r'') para evitar erro de escape no Windows
caminho_salvar = r'C:\Users\ricar\Documents\projetos\social-wave-campaign-optimization\data\resumo_por_canal.pkl'
resumo.to_pickle(caminho_salvar)
print(f'✅ Resumo salvo: {caminho_salvar}')

print('\n' + '=' * 60)
print('TABELA EXECUTIVA COMPLETA!')
print('=' * 60)

📊 TABELA EXECUTIVA POR CANAL

------------------------------------------------------------
MAPEAMENTO DE CANAIS
------------------------------------------------------------
✅ Canal_Nome criado:
   Canal1 -> Meta Ads
   Canal2 -> Google Search
   Canal3 -> YouTube Ads
   Canal4 -> Twitter Ads
   Canal5 -> TikTok
   Canal6 -> LinkedIn Ads

------------------------------------------------------------
SHARES DE GASTO E CONVERSÃO
------------------------------------------------------------
✅ Share_Gasto: % do investimento total por canal
✅ Share_Conversao: % das conversões totais por canal
✅ Diferenca_Share: desbalanceamento (positivo = sobre-investido)

------------------------------------------------------------
TABELA EXECUTIVA
------------------------------------------------------------

📋 Ordenado por CPA (menor = mais eficiente):
               Gasto_Total  Conversao_Total   CPA   CTR  Taxa_Conversao  Share_Gasto  Share_Conversao  Diferenca_Share
Canal_Nome                            